# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toBESkiii/FlyRank_AI_Intership/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task

My chosen lane is **Refresh / Content Opportunity Scoring**.

I will frame this primarily as a **ranking and scoring task** because the main decision is not simply whether a page is good or bad. The decision is which pages an SEO specialist or content editor should investigate first.

The planned system will assign each page a review-priority score and arrange the pages from highest to lowest priority. Pages with stronger evidence of a meaningful content opportunity will appear nearer the top of the review queue.

A supervised classification model may later be used to estimate the probability of an observed outcome, such as decline. However, the probability will be used as one part of a ranking system. Therefore, the final business output is a ranked and scored review queue rather than only a yes-or-no classification.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "toBESkiii/FlyRank_AI_Intership/main/"
    "data/raw/content_refresh_anonymized.csv"
)

page_dataset = pd.read_csv(DATA_URL)

print("Starter dataset loaded successfully.")
print("Number of pages:", page_dataset.shape[0])
print("Number of available columns:", page_dataset.shape[1])
print("Planned ML task type: Ranking / scoring")
print("Planned output: An ordered page-review queue")

Starter dataset loaded successfully.
Number of pages: 30000
Number of available columns: 44
Planned ML task type: Ranking / scoring
Planned output: An ordered page-review queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


The ideal target for this project would be an **observed future outcome**, such as whether a page experiences a meaningful decline or content opportunity during a later evaluation period.

For example, the system could use information from a previous 90-day period to score the likelihood that a page will require review during the following 30 days. This would separate the information used to make the prediction from the future outcome being predicted.

The starter dataset does not currently provide a separate future outcome window. Therefore, for this initial framing exercise, I will create a temporary proxy target called `proxy_decline_target`.

A page will receive:

* `1` when its recorded `trend_direction` is `down`
* `0` when its recorded `trend_direction` is not `down`

This proxy represents an observed downward pattern in the current starter snapshot. However, it is defined from an existing rule rather than measured in a separate future period, so it should not be treated as the final capstone target.

The columns `trend_direction` and `trend_pct` must not be used as model input features when predicting this proxy because they directly contain the information used to create it. Doing so would cause target leakage.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a separate dataframe for the framing exercise
page_target_dataset = page_dataset.copy()

# Create a temporary binary proxy target
# 1 means the page is labelled as having a downward trend
# 0 means the page is not labelled as having a downward trend
page_target_dataset["proxy_decline_target"] = (
    page_target_dataset["trend_direction"] == "down"
).astype(int)

# Count the number of pages in each target group
target_counts = (
    page_target_dataset["proxy_decline_target"]
    .value_counts()
    .sort_index()
)

print("Proxy target distribution:")
print("0 - Not labelled as declining:", target_counts.get(0, 0))
print("1 - Labelled as declining:", target_counts.get(1, 0))

# Show what the target looks like for a few pseudonymised pages
page_target_dataset[
    ["content_id", "trend_direction", "proxy_decline_target"]
].head(10)

Proxy target distribution:
0 - Not labelled as declining: 13738
1 - Labelled as declining: 16262


,content_id,trend_direction,proxy_decline_target
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric for this project will be **Precision@50**.

Precision@50 measures the proportion of the 50 highest-ranked pages that match the defined target or proxy. This metric is appropriate because an SEO specialist or content editor may only have enough time to inspect a limited number of pages. Therefore, the quality of the pages at the top of the queue matters more than general accuracy across all 30,000 pages.

For this initial framing, I will define a provisional success level of **Precision@50 equal to or greater than 0.70**. This would mean that at least 35 of the top 50 recommended pages match the defined review proxy.

The system should also perform better than a transparent fixed-rule baseline. The metric should eventually be calculated on pages from held-out clients rather than on the same data used to create or train the scoring method.

This threshold is provisional because the current proxy is based on an observed trend category rather than a separate future outcome. It may be revised when a stronger future-window target becomes available.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Number of pages an SEO specialist can review
review_capacity = 50

# Provisional minimum number of useful pages in the top 50
minimum_useful_pages = 35

# Calculate the proposed Precision@50 success threshold
target_precision_at_50 = minimum_useful_pages / review_capacity

# Calculate how common the current proxy target is in the full dataset
overall_proxy_rate = (
    page_target_dataset["proxy_decline_target"].mean()
)

print("Review capacity:", review_capacity, "pages")
print("Minimum useful pages required:", minimum_useful_pages)

print(
    "Provisional Precision@50 target:",
    f"{target_precision_at_50:.2f}"
)

print(
    "Overall proxy decline rate:",
    f"{overall_proxy_rate:.2f}"
)

Review capacity: 50 pages
Minimum useful pages required: 35
Provisional Precision@50 target: 0.70
Overall proxy decline rate: 0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


The unit of analysis for this project is **one pseudonymised website content page**.

Each row in the starter dataframe represents one page. The columns contain measurable information about that page, such as its search visibility, clicks, click-through rate, search demand, content characteristics and observed performance trend.

The model or scoring system will evaluate pages individually and assign each page a review-priority score. The pages can then be sorted from highest to lowest priority to create a review queue for an SEO specialist or content editor.

The identifiers are used only to distinguish pages and clients. They should not be treated as predictive features because their values do not describe the quality or performance of the page.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Choose useful columns that are actually available in the dataset
candidate_columns = [
    "client_id",
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "word_count",
    "trend_direction",
    "proxy_decline_target"
]

available_columns = [
    column
    for column in candidate_columns
    if column in page_target_dataset.columns
]

# Create a small page-level dataframe for display
page_level_sample = page_target_dataset[
    available_columns
].head(5).copy()

print("Unit of analysis: one pseudonymised content page")
print("Number of rows in the full dataset:", len(page_target_dataset))
print("Columns displayed:", available_columns)

display(page_level_sample)

Unit of analysis: one pseudonymised content page
Number of rows in the full dataset: 30000
Columns displayed: ['client_id', 'content_id', 'impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'search_volume', 'word_count', 'trend_direction', 'proxy_decline_target']


,client_id,content_id,impressions_90d,clicks_90d,ctr,avg_position,search_volume,word_count,trend_direction,proxy_decline_target
0,client_f369cb89fc,content_304f48230142,3803,29,0.76,10.6,10.0,3221.0,down,1
1,client_4e07408562,content_a1fb4e703a9e,15320,7,0.05,20.3,90.0,2481.0,down,1
2,client_7f2253d7e2,content_9aa793d4d895,12581,11,0.09,36.5,0.0,3515.0,down,1
3,client_19581e27de,content_331d6c4de07b,11751,58,0.49,6.2,10.0,NaN,stable,0
4,client_3fdba35f04,content_d99b7a2d90ca,19140,24,0.13,44.0,0.0,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A transparent fixed rule should be used as the baseline for this project.

For example, a simple rule could prioritise pages that have at least 100 impressions, a below-median click-through rate and an average search position within the top 20. This rule is easy to understand and provides a useful starting point.

However, a fixed rule may be too limited because page opportunities can depend on several signals interacting with one another. A page with low CTR may not require the same action as another page with low CTR. Its search position, impressions, content age, word count, engagement and recent performance may change how urgently it should be reviewed.

A fixed rule also uses manually chosen thresholds. Pages just above or below a threshold may be treated very differently even when their characteristics are similar.

Machine learning may improve the ranking by learning combinations and relative importance of several features from the available data. It may identify useful review candidates that a simple rule misses and assign more gradual priority scores instead of only producing a yes-or-no result.

However, machine learning only earns its place if it performs better than the fixed-rule baseline on held-out data and still produces recommendations that a human can understand. The goal is not to use a complicated model simply because one is available. The goal is to create a more useful and reliable ranked review queue.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create a transparent fixed-rule baseline

median_ctr = page_target_dataset["ctr"].median()

fixed_rule_mask = (
    (page_target_dataset["impressions_90d"] >= 100)
    & (page_target_dataset["ctr"] < median_ctr)
    & (page_target_dataset["avg_position"] <= 20)
)

fixed_rule_candidates = page_target_dataset[
    fixed_rule_mask
].copy()

print("Median CTR used by the fixed rule:", round(median_ctr, 4))
print("Pages selected by the fixed rule:", len(fixed_rule_candidates))

# Check how many selected pages match the temporary proxy target
fixed_rule_precision = (
    fixed_rule_candidates["proxy_decline_target"].mean()
    if len(fixed_rule_candidates) > 0
    else 0
)

print(
    "Proportion matching the decline proxy:",
    f"{fixed_rule_precision:.2f}"
)

display(
    fixed_rule_candidates[
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "proxy_decline_target"
        ]
    ].head(10)
)

Median CTR used by the fixed rule: 0.07
Pages selected by the fixed rule: 3875
Proportion matching the decline proxy: 0.74


,content_id,impressions_90d,ctr,avg_position,proxy_decline_target
5,content_d4084a4bc775,3970,0.03,8.5,1
33,content_d87a116e2c79,298,0.00,6.8,0
34,content_55f75c034970,3998,0.03,6.4,0
41,content_e45a618f6b32,231,0.00,11.0,1
43,content_1938955b34c4,184,0.00,2.9,1
45,content_2a6383ed421f,691,0.00,10.8,1
50,content_fe4e6c1a622c,6971,0.01,11.9,1
52,content_a4cd54c0a5f1,416,0.00,10.8,1
54,content_ff8ea1364b59,170,0.00,10.7,1
66,content_d512773c9590,116,0.00,6.8,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.